In [0]:
# Crio os schemas que vão armazenar os registros tratados
# e os registros reprovados pelas regras de qualidade.

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.silver
    COMMENT 'Camada Silver com dados da ANP tratados, padronizados e aprovados pelas regras de qualidade.'
""")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.quarentena
    COMMENT 'Registros da ANP que não passaram pelas regras de qualidade e precisam ser preservados para análise.'
""")

print("Schemas Silver e Quarentena disponíveis.")

In [0]:
display(
    spark.sql("SHOW SCHEMAS IN workspace")
)

In [0]:
# A Silver parte sempre da tabela Bronze persistida no Unity Catalog.
# Dessa forma este notebook não depende da execução do notebook anterior.

nome_tabela_bronze = "workspace.bronze.revendas_anp"

df_bronze = spark.table(nome_tabela_bronze)

print("Registros recebidos da Bronze:", df_bronze.count())
print("Colunas recebidas:", len(df_bronze.columns))

In [0]:
from pyspark.sql.functions import col

# Na Silver padronizo os nomes para snake_case.
# A Bronze continua preservando exatamente a estrutura recebida da ANP.
df_silver_base = df_bronze.select(
    col("CNPJ").alias("cnpj"),
    col("RAZÃO").alias("razao_social"),
    col("FANTASIA").alias("nome_fantasia"),
    col("ENDEREÇO").alias("endereco"),
    col("NÚMERO").alias("numero"),
    col("COMPLEMENTO").alias("complemento"),
    col("BAIRRO").alias("bairro"),
    col("CEP").alias("cep"),
    col("MUNICÍPIO").alias("municipio"),
    col("ESTADO").alias("estado"),
    col("BANDEIRA").alias("bandeira"),
    col("PRODUTO").alias("produto"),
    col("UNIDADE DE MEDIDA").alias("unidade_medida"),
    col("PREÇO DE REVENDA").alias("preco_revenda"),
    col("DATA DA COLETA").alias("data_coleta"),

    # Mantenho a linha original para conseguir rastrear
    # qualquer registro até o arquivo Excel recebido.
    col("linha_origem"),

    # Preservo os metadados técnicos da ingestão.
    col("id_execucao"),
    col("id_arquivo"),
    col("arquivo_origem"),
    col("data_ingestao_utc"),
    col("hash_sha256")
)

print("Registros na base Silver:", df_silver_base.count())
print("Colunas:", len(df_silver_base.columns))

In [0]:
# Reutilizo as mesmas regras de qualidade que já foram desenvolvidas e validadas no pipeline local.

from src.qualidade.validar_cnpj import (
    limpar_cnpj,
    normalizar_cnpj,
    validar_cnpj,
)

from src.qualidade.validar_cep import (
    classificar_cep,
    normalizar_cep,
    validar_formato_cep
)

print("Módulos de qualidade importados com sucesso.")

In [0]:
display(
    df_silver_base.limit(10)
)

In [0]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, BooleanType

# Reutilizo as mesmas regras de CNPJ validadas no pipeline local.
# Neste momento mantenho as colunas intermediárias para conseguir auditar cada etapa da transformação.

udf_limpar_cnpj = udf(limpar_cnpj, StringType())
udf_normalizar_cnpj = udf(normalizar_cnpj, StringType())
udf_validar_cnpj = udf(validar_cnpj, BooleanType())

df_qualidade = (
    df_silver_base
    .withColumn("cnpj_original", col("cnpj"))
    .withColumn("cnpj_limpo", udf_limpar_cnpj(col("cnpj")))
    .withColumn(
        "cnpj_normalizado",
        udf_normalizar_cnpj(col("cnpj"))
    )
    .withColumn(
        "cnpj_valido",
        udf_validar_cnpj(col("cnpj_normalizado"))
    )
)

print("Registros após validação do CNPJ:", df_qualidade.count())

In [0]:
display(
    df_qualidade.select(
        "linha_origem",
        "cnpj_original",
        "cnpj_limpo",
        "cnpj_normalizado",
        "cnpj_valido"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import when, count
# Aqui conto quantos registros passaram ou não pela validação do CNPJ.
display(
    df_qualidade.select(
        count(
            when(col("cnpj_valido") == True,1)
        ).alias("cnpjs_validos"),

        count(
            when(col("cnpj_valido") == False, 1)
        ).alias("cnpjs_invalidos")
    )
)

In [0]:
# Aplico as mesmas regras de qualidade de CEP usadas no pipeline local.
# Preservo o valor original e acompanho classificação e normalização.

udf_classificar_cep = udf(classificar_cep, StringType())
udf_normalizar_cep = udf(normalizar_cep, StringType())
udf_validar_formato_cep = udf(validar_formato_cep, BooleanType())

df_qualidade = (
    df_qualidade
    .withColumn("cep_original", col("cep"))
    .withColumn(
        "status_cep",
        udf_classificar_cep(col("cep"))
    )
    .withColumn(
        "cep_normalizado",
        udf_normalizar_cep(col("cep"))
    )
    .withColumn(
        "cep_formato_valido",
        udf_validar_formato_cep(col("cep"))
    )
)

print("Registros após validação do CEP:", df_qualidade.count())

In [0]:
display(
    df_qualidade
    .groupBy("status_cep")
    .count()
    .orderBy("status_cep")
)

In [0]:
# Confirmo quantos CEPs podem seguir após a validação estrutural

display(
    df_qualidade
    .groupBy("cep_formato_valido")
    .count()
    .orderBy("cep_formato_valido")
)

In [0]:
# Antes de separar a quarentena, identifico exatamente
# qual registro não passou pela validação do CEP.

display(
    df_qualidade
    .filter(col("cep_formato_valido") == False)
    .select(
        "linha_origem",
        "cnpj_original",
        "razao_social",
        "cep_original",
        "cep_normalizado",
        "status_cep",
        "municipio",
        "estado",
        "produto"
    )
)

In [0]:
from pyspark.sql.functions import when, lit, concat_ws

# Classifico cada registro conforme as regras críticas de qualidade.
# Registros com CNPJ válido e CEP aceitável seguem para a Silver.
df_qualidade = (
    df_qualidade
    .withColumn(
        "status_qualidade",
        when(
            (col("cnpj_valido") == True)
            & (col("cep_formato_valido") == True),
            lit("APROVADO")
        ).otherwise(lit("QUARENTENA"))
    )
    .withColumn(
        "motivo_quarentena",
        when(
            col("status_qualidade") == "QUARENTENA",
            concat_ws(
                "|",
                when(
                    col("cnpj_valido") == False,
                    lit("CNPJ_INVALIDO")
                ),
                when(
                    col("cep_formato_valido") == False,
                    lit("CEP_SUSPEITO")
                )
            )
        )
    )
)

In [0]:
display(
    df_qualidade
    .groupBy(
        "status_qualidade",
        "motivo_quarentena"
    )
    .count()
    .orderBy("status_qualidade")
)

In [0]:
# Separo os registros aprovados, que seguirão para a camada Silver.

df_silver = (
    df_qualidade.filter(col("status_qualidade") == "APROVADO")
)

# Preservo separadamente os registros  rejeitados para auditoria.
df_quarentena = (
    df_qualidade
    .filter(col("status_qualidade") == "QUARENTENA")
)

print("Registros Silver:", df_silver.count())
print("Registros Quarentena:", df_quarentena.count())

In [0]:
display(
    df_quarentena.select(
        "linha_origem",
        "cnpj_original",
        "razao_social",
        "cep_original",
        "cep_normalizado",
        "status_cep",
        "municipio",
        "estado",
        "produto",
        "status_qualidade",
        "motivo_quarentena"
    )
)

In [0]:
from pyspark.sql.functions import current_timestamp

# Na Silver final mantenho apenas os registros aprovados
# e uso os identificadores já normalizados.
df_silver_final = (
    df_silver
    .select(
        col("cnpj_normalizado").alias("cnpj"),
        col("razao_social"),
        col("nome_fantasia"),
        col("endereco"),
        col("numero"),
        col("complemento"),
        col("bairro"),
        col("cep_normalizado").alias("cep"),
        col("municipio"),
        col("estado"),
        col("bandeira"),
        col("produto"),
        col("unidade_medida"),
        col("preco_revenda"),
        col("data_coleta"),
        col("linha_origem"),
        col("id_execucao"),
        col("id_arquivo"),
        col("arquivo_origem"),
        col("hash_sha256"),
        current_timestamp().alias("data_processamento_utc")
    )
)

print("Registros Silver final:", df_silver_final.count())
print("Colunas Silver final:", len(df_silver_final.columns))

In [0]:
# Na Quarentena preservo os valores problemáticos, as etapas de validação
# e os metadados necessários para auditoria e possível reprocessamento.
df_quarentena_final = (
    df_quarentena
    .select(
        col("linha_origem"),

        col("cnpj_original"),
        col("cnpj_limpo"),
        col("cnpj_normalizado"),
        col("cnpj_valido"),

        col("razao_social"),
        col("nome_fantasia"),
        col("endereco"),
        col("numero"),
        col("complemento"),
        col("bairro"),

        col("cep_original"),
        col("cep_normalizado"),
        col("status_cep"),
        col("cep_formato_valido"),

        col("municipio"),
        col("estado"),
        col("bandeira"),
        col("produto"),
        col("unidade_medida"),
        col("preco_revenda"),
        col("data_coleta"),

        col("status_qualidade"),
        col("motivo_quarentena"),

        col("id_execucao"),
        col("id_arquivo"),
        col("arquivo_origem"),
        col("hash_sha256"),

        current_timestamp().alias("data_processamento_utc")
    )
)

print("Registros Quarentena final:", df_quarentena_final.count())
print("Colunas Quarentena final:", len(df_quarentena_final.columns))

In [0]:
# Faço uma última validação antes de persistir os dados no Unity Catalog.

print("Silver")
print("Registros:", df_silver_final.count())
print("Colunas:", len(df_silver_final.columns))

print()

print("Quarentena")
print("Registros:", df_quarentena_final.count())
print("Colunas:", len(df_quarentena_final.columns))

In [0]:
# Confirmo que nenhum registro foi perdido durante a separação.

total_processado = (
    df_silver_final.count()
    + df_quarentena_final.count()
)

print("Total recebido da Bronze:", df_bronze.count())
print("Total após qualidade:", total_processado)

In [0]:
nome_tabela_silver = "workspace.silver.revendas_anp"

# Gravo a Silver como tabela Delta gerencada pelo Unity Catalog.
(
    df_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_tabela_silver)
)

print("Tabela Silver gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT hash_sha256) AS arquivos_distintos,
            MIN(linha_origem) AS primeira_linha,
            MAX(linha_origem) AS ultima_linha
        FROM workspace.silver.revendas_anp
    """)
)

In [0]:
nome_tabela_quarentena = "workspace.quarentena.revendas_anp"

# Gravo os registros rejeitados em uma tabela Delta separada para manter auditoria e permitir análise ou reprocessamento.
(
    df_quarentena_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_tabela_quarentena)
)

print("Tabela Quarentena gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            MIN(linha_origem) AS linha_origem,
            MAX(motivo_quarentena) AS motivo_quarentena
        FROM workspace.quarentena.revendas_anp
    """)
)